# Session 02: Python Environments with `uv`

Today we are solving a very practical problem: how do we make sure everyone in the course is running the same Python setup?

Students often start by installing packages wherever Python happens to be installed. That works for a while, then something breaks: one notebook cannot find `pandas`, another notebook uses the wrong package version, or Jupyter is running a different Python than Terminal.

A Python environment fixes this by giving one project its own Python setup and package list.


## Learning objectives

By the end of this session, you should be able to:

- Explain what a Python environment is.
- Explain why environments are useful for data analysis.
- Create the course environment with `uv`.
- Install the course packages from `pyproject.toml`.
- Register the course environment as a Jupyter kernel.
- Check which Python your notebook is using.


## The simplest explanation

A Python environment is a project-specific Python workspace.

It answers two questions:

- Which Python should run this code?
- Which packages should be available?

For this course, we want one predictable environment with the packages we will use: `pandas`, `numpy`, `plotly`, `scikit-learn`, `requests`, `yfinance`, `streamlit`, and Jupyter tools.


## Why not just install everything globally?

Global installs feel simpler at the beginning, but they create hidden problems:

| Without environments | With environments |
| --- | --- |
| Every project shares the same packages. | Each project has its own packages. |
| Updating one package can break old work. | A project keeps the versions it expects. |
| Jupyter may use the wrong Python. | We can select the course kernel explicitly. |
| It is hard to reproduce another student's setup. | Everyone can recreate the same setup from `pyproject.toml`. |

The point is not to make Python more complicated. The point is to make mistakes easier to diagnose.


## Diagram 1: laptop, base Python, and project environment

Your laptop can have more than one Python available.

The **base Python** is the general Python installation on the computer. A **project environment** is a separate Python workspace created for one project or course.

```mermaid
flowchart TD
    laptop["Your laptop"] --> os["Operating system"]
    os --> base["Base Python installation"]
    os --> project["Course project folder"]
    project --> pyproject["pyproject.toml
Package list"]
    project --> venv[".venv
Course environment"]
    venv --> interpreter["Python interpreter
inside .venv"]
    venv --> packages["Installed packages
pandas, numpy, plotly, ..."]
```

The important idea: the project environment lives on the same laptop, but it is not the same thing as the base Python installation.

## Diagram 2: what `uv sync` creates

`uv sync` reads the project configuration and prepares the environment.

```mermaid
flowchart LR
    pyproject["pyproject.toml
Direct dependencies
Example: pandas"] --> sync["uv sync"]
    sync --> resolver["uv resolver
Chooses compatible versions"]
    resolver --> lock["uv.lock
Generated exact dependency map
Direct + indirect packages"]
    lock --> venv[".venv
Installed local environment"]
    venv --> python["Python interpreter"]
    venv --> libs["Installed libraries
Example: pandas + numpy + python-dateutil"]
```

`pyproject.toml` is the recipe that humans usually edit. It lists the packages the project directly needs.

`uv.lock` is generated by `uv`. It records the exact package versions that were chosen, including packages you did not type yourself but are required by your packages. For example, if the project needs `pandas`, the lock file may also record packages that `pandas` depends on.

The practical idea: `pyproject.toml` says what the project needs; `uv.lock` says exactly what was installed.

## Diagram 3: Jupyter, kernels, and interpreters

A notebook does not run Python by itself. It sends code to a **kernel**. The kernel points to a Python interpreter.

```mermaid
flowchart LR
    notebook["Notebook
.ipynb file"] --> kernel["Selected Jupyter kernel"]
    kernel --> interpreter["Python interpreter"]
    interpreter --> packages["Available packages"]
```

When we register the course environment with `ipykernel`, Jupyter can show it as `Python (PDA2 course)`. Selecting that kernel makes the notebook use the Python interpreter inside the course environment.

## Diagram 4: the wrong-kernel problem

This is one of the most common beginner problems.

```mermaid
flowchart TD
    notebook["Notebook"] --> selected["Selected kernel"]
    selected --> wrong["Base Python"]
    wrong --> missing["Package missing
ModuleNotFoundError"]

    course["Course environment
.venv"] --> installed["pandas installed here"]
```

The package may be installed correctly in `.venv`, but the notebook fails because Jupyter is using another Python. This is why we check `sys.executable` and select `Python (PDA2 course)`.

## The two files in this folder

This session folder should contain only:

```text
s02_python_environments_uv/
├── pyproject.toml
└── s02_python_environments_uv.ipynb
```

`pyproject.toml` contains the package list for the course environment.

This notebook explains how to create and use that environment.


## Step 1: open Terminal in this folder

In Terminal, move into this session folder:

```bash
cd /Users/dgh/Desktop/pda_mbads/pda2/class_material/s02_python_environments_uv
```

Check that you are in the correct folder:

```bash
pwd
ls
```

You should see `pyproject.toml` and this notebook.


## Managing two environments in this repository

This repository has two `pyproject.toml` files:

| Location | Purpose |
| --- | --- |
| `/Users/dgh/Desktop/pda_mbads/pyproject.toml` | Environment for the whole course repository. |
| `/Users/dgh/Desktop/pda_mbads/pda2/class_material/s02_python_environments_uv/pyproject.toml` | Small environment used to teach this session. |

That is okay. The important rule is: **the folder you run uv from decides which project uv uses**.

```mermaid
flowchart TD
    laptop["Laptop"] --> repo["/Users/dgh/Desktop/pda_mbads"]
    repo --> root_toml["Root pyproject.toml"]
    repo --> root_venv["Root .venv"]
    repo --> s02["pda2/class_material/s02_python_environments_uv"]
    s02 --> s02_toml["s02 pyproject.toml"]
    s02 --> s02_venv["s02 .venv"]
```

If Terminal is here:

```bash
cd /Users/dgh/Desktop/pda_mbads
uv sync
```

then uv uses the root `pyproject.toml` and creates or updates the root `.venv`.

If Terminal is here:

```bash
cd /Users/dgh/Desktop/pda_mbads/pda2/class_material/s02_python_environments_uv
uv sync
```

then uv uses the `s02` `pyproject.toml` and creates or updates the `s02` `.venv`.

When in doubt, use `pwd` before running uv.

You can also make the project explicit with `--project`:

```bash
uv --project /Users/dgh/Desktop/pda_mbads run python --version
```

```bash
uv --project /Users/dgh/Desktop/pda_mbads/pda2/class_material/s02_python_environments_uv run python --version
```

This is useful when you want to be very clear about which environment is being used.

## Activating different environments

You do not have to activate an environment if you use `uv run`. These two commands are usually enough:

```bash
uv sync
uv run python
```

But activation is useful for teaching because students can see that the active Python changes.

### Activate the root repo environment

macOS/Linux:

```bash
cd /Users/dgh/Desktop/pda_mbads
uv sync
source .venv/bin/activate
which python
python --version
deactivate
```

Windows PowerShell:

```powershell
cd C:\path\to\pda_mbads
uv sync
.venv\Scripts\Activate.ps1
where python
python --version
deactivate
```

### Activate the `s02` environment

macOS/Linux:

```bash
cd /Users/dgh/Desktop/pda_mbads/pda2/class_material/s02_python_environments_uv
uv sync
source .venv/bin/activate
which python
python --version
deactivate
```

Windows PowerShell:

```powershell
cd C:\path\to\pda_mbads\pda2\class_material\s02_python_environments_uv
uv sync
.venv\Scripts\Activate.ps1
where python
python --version
deactivate
```

The output of `which python` or `where python` should point inside the `.venv` folder of the environment you activated.

## Showing both environments in Jupyter

To make the difference visible in notebooks, register each environment with a different Jupyter kernel name.

Root repo environment:

```bash
cd /Users/dgh/Desktop/pda_mbads
uv sync
uv run python -m ipykernel install --user --name pda-mbads --display-name "Python (PDA MBADS repo)"
```

`s02` environment:

```bash
cd /Users/dgh/Desktop/pda_mbads/pda2/class_material/s02_python_environments_uv
uv sync
uv run python -m ipykernel install --user --name pda2-course --display-name "Python (PDA2 course)"
```

Then Jupyter should show two different kernel options:

- `Python (PDA MBADS repo)`
- `Python (PDA2 course)`

This is the cleanest way to demonstrate that a notebook can use different Python environments.

## Check the current folder from the notebook

This is not creating anything. It only shows where this notebook is currently running from.


In [ ]:
# Show the folder used by this notebook.
from pathlib import Path

Path.cwd()


## Step 2: install `uv`

Most students will not have `uv` installed yet. Install it once on your laptop, then use it throughout the course.

### macOS

Open Terminal and run:

```bash
curl -LsSf https://astral.sh/uv/install.sh | sh
```

If you use Homebrew, this also works:

```bash
brew install uv
```

### Windows

Open PowerShell and run:

```powershell
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
```

If you use WinGet, this also works:

```powershell
winget install --id=astral-sh.uv -e
```

After installing, close Terminal or PowerShell and open it again. This matters because your shell may need to reload its `PATH` before it can find `uv`.

Then verify the installation:

```bash
uv --version
```

You should see a version number.

You can also check `uv` from a notebook cell after installation.

If this fails, finish the Terminal or PowerShell installation first, restart Jupyter, and run the cell again.

In [ ]:
# Check whether uv is available.
!uv --version


## Step 3: `uv init` vs `uv sync`

These two commands are used at different moments.

| Situation | Command | Why |
| --- | --- | --- |
| You are starting a brand-new project with no `pyproject.toml`. | `uv init` | Creates the first project files. |
| You already have a project folder with `pyproject.toml`. | `uv sync` | Builds the environment from the existing project file. |

For this course folder, we already have a `pyproject.toml`. That means we do **not** use `uv init` here.

We use:

```bash
uv sync
```

What `uv sync` does:

- Reads `pyproject.toml`.
- Creates a local `.venv` folder if needed.
- Installs the packages listed for this course.
- Creates or updates `uv.lock` with the exact resolved package versions, including indirect dependencies.

Use this rule:

```text
No pyproject.toml yet?  -> uv init
Already has pyproject.toml? -> uv sync
```

The course `pyproject.toml` includes three notebook-related packages:

| Package | Why we need it |
| --- | --- |
| `jupyter` | Provides the Jupyter notebook tools. |
| `notebook` | Lets students open and run `.ipynb` files. |
| `ipykernel` | Lets this uv environment appear as a selectable Jupyter kernel. |

After running `uv sync`, it is normal to see `.venv/` and `uv.lock`. We are not storing them as class material, but your computer may create them locally.

## Understanding `pyproject.toml`

`pyproject.toml` is the project file that tells Python tools what this project is and what packages it needs.

In this course folder, it contains:

- The project name.
- The required Python version.
- The list of course dependencies.

When you run `uv sync`, `uv` reads this file and builds the local environment from it.

In [ ]:
# Show the project configuration file.
Path("pyproject.toml").read_text()

The most important part is the `dependencies` list.

Each line is one package that the course environment should contain. For example, `pandas` is needed for dataframes, `plotly` is needed for visualizations, and `ipykernel` is needed so Jupyter can use this environment as a kernel.

## Installing dependencies with `uv add`

Use `uv add` when a project needs a package that is not yet listed in `pyproject.toml`.

Example:

```bash
uv add seaborn
```

What this does:

- Adds `seaborn` to `pyproject.toml`.
- Installs it into `.venv`.
- Updates `uv.lock`.

For this course environment, do not add random packages unless an exercise asks for them. The goal is to keep everyone using the same setup.

## Removing dependencies with `uv remove`

Use `uv remove` when a project no longer needs a package.

Example:

```bash
uv remove seaborn
```

What this does:

- Removes `seaborn` from `pyproject.toml`.
- Updates the environment.
- Updates `uv.lock`.

Do not delete package lines manually unless you know what you are doing. Let `uv` update the project files consistently.

## Removing a project environment

Removing a dependency and removing an environment are different actions.

| Action | What changes |
| --- | --- |
| `uv remove package_name` | Removes one package from the project. |
| Delete `.venv` | Removes the whole local environment folder. |

For a uv project, the environment usually lives in a folder called `.venv`.

There is no separate `uv env remove` command for a normal project environment. To remove the environment, delete `.venv` from the project folder.

macOS/Linux:

```bash
rm -r .venv
```

Windows PowerShell:

```powershell
Remove-Item -Recurse .venv
```

After removing it, the project is not lost. The important project file is still `pyproject.toml`.

To recreate the environment:

```bash
uv sync
```

This is why environments are disposable: you can delete `.venv` and rebuild it from the project files.

## Reproducing a project setup on another machine

This is one of the main reasons environments matter.

If another student or instructor gives you a project with a `pyproject.toml`, you do not need to guess which packages to install.

Use this workflow:

```bash
cd path/to/project
uv sync
uv run jupyter notebook
```

Then select the correct Jupyter kernel.

For this course:

```bash
uv run python -m ipykernel install --user --name pda2-course --display-name "Python (PDA2 course)"
```

Important distinction:

| File | Role in reproduction |
| --- | --- |
| `pyproject.toml` | Says what the project needs. |
| `uv.lock` | Records the exact resolved versions for direct and indirect dependencies. |

For simple class setup, `pyproject.toml` is enough to create the environment. For professional projects shared across a team, you normally also share `uv.lock` so everyone gets the exact same resolved versions, not just compatible versions.

## Step 4: prove the environment works

Run this in Terminal:

```bash
uv run python --version
```

Then run:

```bash
uv run python -c "import pandas; print(pandas.__version__)"
```

`uv run` means: run this command using the environment for this project.


## Step 5: register the environment in Jupyter

This step is necessary if you want to open Jupyter normally and choose this uv environment from the kernel menu. If you also use the root repo environment, give each kernel a different display name.

It is not strictly necessary if you always start Jupyter from this folder with:

```bash
uv run jupyter notebook
```

However, registering the kernel is recommended for the course because it makes the environment easy to select in Jupyter, VS Code, and other notebook tools.

Jupyter uses kernels. A kernel is the Python environment that runs notebook code.

This is where `ipykernel` matters. Installing `ipykernel` does not automatically select the environment for every notebook. It gives Python the command needed to register this uv environment as a Jupyter kernel.

Run this once in Terminal:

```bash
uv run python -m ipykernel install --user --name pda2-course --display-name "Python (PDA2 course)"
```

What the command means:

| Part | Meaning |
| --- | --- |
| `uv run python` | Use the Python interpreter from this uv environment. |
| `-m ipykernel install` | Register that interpreter as a Jupyter kernel. |
| `--user` | Install the kernel for your user account. |
| `--name pda2-course` | Internal kernel name. |
| `--display-name "Python (PDA2 course)"` | Name students see in Jupyter. |

Then, in Jupyter:

1. Open a notebook.
2. Click the kernel selector.
3. Choose `Python (PDA2 course)`.
4. Restart the notebook kernel.
5. Run the checks below.

## Selecting environments in different notebook tools

The setup step is the same for all tools:

```bash
uv run python -m ipykernel install --user --name pda2-course --display-name "Python (PDA2 course)"
```

After this command, `Python (PDA2 course)` becomes a kernel option.

### JupyterLab

Start JupyterLab from the environment:

```bash
uv run jupyter lab
```

Then choose the environment:

- New notebook: use the Launcher and select `Python (PDA2 course)`.
- Existing notebook: open the notebook, click the kernel name near the top-right area, then choose `Python (PDA2 course)`.
- If the kernel does not appear, refresh the browser page or restart JupyterLab.

### Classic Jupyter Notebook

Start classic Notebook from the environment:

```bash
uv run jupyter notebook
```

Then choose the environment:

- New notebook: `New` > `Python (PDA2 course)`.
- Existing notebook: `Kernel` > `Change kernel` > `Python (PDA2 course)`.

### VS Code

Open the notebook in VS Code.

Then choose the environment:

- Click `Select Kernel` in the top-right of the notebook.
- Choose `Python (PDA2 course)` if it appears.
- If it does not appear, choose `Python Environments` and select the `.venv` inside this project folder.
- If VS Code still does not find it, run the `ipykernel install` command again from the project folder.

### Google Colab

Google Colab is different from local Jupyter tools.

In a normal hosted Colab notebook, the code runs on Google's runtime, not on your laptop. That means Colab cannot directly select your local uv `.venv` as a kernel.

You have two options.

#### Option A: use the hosted Colab runtime

Use this when you do not need the local course environment.

Install packages inside the Colab notebook:

```python
!pip install pandas numpy plotly scikit-learn requests yfinance
```

This is convenient, but it is not the same as using the uv environment on your laptop. Packages installed this way belong to the temporary Colab runtime.

#### Option B: connect Colab to your local uv environment

Use this when you want Colab's interface but want code to run on your laptop using your uv environment.

First, run this from the project folder in Terminal:

```bash
cd /Users/dgh/Desktop/pda_mbads/pda2/class_material/s02_python_environments_uv
uv sync
uv run jupyter notebook \
  --NotebookApp.allow_origin='https://colab.research.google.com' \
  --port=8888 \
  --NotebookApp.port_retries=0 \
  --NotebookApp.allow_credentials=True
```

The terminal will show a local URL that includes a token, usually starting with `http://localhost:8888/?token=...`.

Then, in Colab:

1. Click `Connect`.
2. Choose `Connect to local runtime`.
3. Paste the local URL from Terminal.
4. Click `Connect`.

Security note: only connect trusted notebooks to a local runtime. A Colab notebook connected to your local runtime can run code on your laptop and access local files.

### Terminal

In Terminal, you do not select a kernel. You select the environment by running commands from the correct folder:

```bash
cd /Users/dgh/Desktop/pda_mbads/pda2/class_material/s02_python_environments_uv
uv run python
```

Or by making the project explicit:

```bash
uv --project /Users/dgh/Desktop/pda_mbads/pda2/class_material/s02_python_environments_uv run python
```

The key idea is always the same: make sure the tool is using the Python interpreter from the environment you intended.

## Check which Python this notebook is using

This should point to the environment selected by your Jupyter kernel.

If you do not see the course environment, change the kernel to `Python (PDA2 course)`.


In [ ]:
# Show the Python executable used by this notebook.
import sys

sys.executable


Now check the Python version.


In [ ]:
# Show the Python version used by this notebook.
import platform

platform.python_version()


## Check that course packages are available

If the kernel is correct, these imports should work.


In [ ]:
# Import the main packages for the course.
import numpy
import pandas
import plotly
import requests
import sklearn

"Course packages imported successfully"


Check one package version directly.


In [ ]:
# Show the pandas version in this environment.
pandas.__version__


## How to create a new environment for another project

For a brand new project outside this course folder, the workflow is:

```bash
mkdir my_project
cd my_project
uv init
uv add pandas numpy plotly jupyter ipykernel
uv run python -m ipykernel install --user --name my-project --display-name "Python (my project)"
uv run jupyter notebook
```

If the new project is inside this course repository, use this instead:

```bash
uv init --no-workspace
```

That prevents `uv` from trying to connect the small practice project to the larger course repository.


## The commands you will use most often

| Task | Command |
| --- | --- |
| Create a brand-new project | `uv init` |
| Create the course environment | `uv sync` |
| Run Python inside the environment | `uv run python` |
| Add a package | `uv add package_name` |
| Remove a package | `uv remove package_name` |
| Remove the local environment | `rm -r .venv` on macOS/Linux, `Remove-Item -Recurse .venv` on Windows |
| Start classic Notebook from the environment | `uv run jupyter notebook` |
| Start JupyterLab from the environment | `uv run jupyter lab` |
| Start a local runtime for Colab | `uv run jupyter notebook --NotebookApp.allow_origin='https://colab.research.google.com' --port=8888 --NotebookApp.port_retries=0 --NotebookApp.allow_credentials=True` |
| Register a Jupyter kernel | `uv run python -m ipykernel install --user --name NAME --display-name "DISPLAY NAME"` |
| Recreate a project on another machine | `uv sync` |

For this course, the most important habit is using the `Python (PDA2 course)` kernel in notebooks.

## Troubleshooting

| Problem | Likely cause | Fix |
| --- | --- | --- |
| `uv: command not found` | `uv` is not installed or Terminal cannot find it. | Install `uv`, then restart Terminal. |
| `ModuleNotFoundError` | The package is missing from the active environment. | Run `uv add package_name` or check the kernel. |
| Package works in Terminal but not notebook | Jupyter is using a different Python. | Select `Python (PDA2 course)`. |
| `pyproject.toml` not found | Terminal is in the wrong folder. | Run `pwd` and `ls`. |
| Notebook uses unexpected Python | Wrong kernel selected. | Check `sys.executable`, then switch kernel. |
| Environment seems broken | `.venv` may contain a bad or outdated install. | Delete `.venv`, then run `uv sync`. |


## Knowledge check

Answer these in your own words:

1. What problem does a Python environment solve?
2. What does `pyproject.toml` do?
3. When should you use `uv init`?
4. When should you use `uv sync`?
5. What does `uv add` do?
6. What does `uv remove` do?
7. How do you remove a project environment?
8. How do you select the course environment in JupyterLab or VS Code?
9. Why can hosted Colab not directly select your local uv `.venv`?
10. Why does Jupyter need the correct kernel?
11. How does uv decide which `pyproject.toml` to use?
12. What files help reproduce a project setup on another machine?